In [ ]:
from pymatgen.core.structure import Lattice
import numpy as np
from scipy.optimize import linear_sum_assignment

def reorder_trajectory(structures):
    """Reorders atoms to fix swapping/rotation using Hungarian Algorithm."""
    print("Reordering trajectory...")
    ref_structure = structures[0]
    
    species_indices = {}
    for i, site in enumerate(ref_structure):
        spec = site.specie.symbol
        if spec not in species_indices:
            species_indices[spec] = []
        species_indices[spec].append(i)
        
    sorted_structures = []
    
    for t, struct in enumerate(structures):
        if t == 0:
            sorted_structures.append(struct)
            continue
            
        new_struct = struct.copy()
        for spec, indices in species_indices.items():
            ref_frac = np.array([ref_structure[i].frac_coords for i in indices])
            curr_frac = np.array([struct[i].frac_coords for i in indices])
            dists = struct.lattice.get_all_distances(ref_frac, curr_frac)
            row_ind, col_ind = linear_sum_assignment(dists)
            sorted_frac_subset = curr_frac[col_ind]
            for local_i, global_i in enumerate(indices):
                new_struct[global_i].frac_coords = sorted_frac_subset[local_i]
        sorted_structures.append(new_struct)
    return sorted_structures

def unwrap_trajectory(structures):
    """Unwraps PBC to generate continuous coordinates."""
    print("Unwrapping PBCs...")
    frac_coords = np.array([s.frac_coords for s in structures])
    n_steps, n_atoms, _ = frac_coords.shape
    
    unwrapped_frac = np.zeros_like(frac_coords)
    unwrapped_frac[0] = frac_coords[0]
    image_shifts = np.zeros((n_atoms, 3), dtype=int)
    
    for t in range(1, n_steps):
        diff = frac_coords[t] - frac_coords[t-1]
        jump = np.round(diff)
        image_shifts -= jump.astype(int)
        unwrapped_frac[t] = frac_coords[t] + image_shifts

    return unwrapped_frac

def fold_and_average_supercell(structures, unwrapped_frac, scaling):
    """
    Folds a supercell trajectory back into the unit cell and calculates 
    averaged thermal ellipsoids.
    """
    print(f"Folding {scaling[0]}x{scaling[1]}x{scaling[2]} supercell...")
    
    avg_frac_unwrapped_sc = np.mean(unwrapped_frac, axis=0)
    avg_frac_wrapped_sc = avg_frac_unwrapped_sc % 1.0
    
    unit_frac_coords = avg_frac_wrapped_sc * np.array(scaling)
    unit_frac_coords = unit_frac_coords % 1.0
    
    sc_lattice = structures[0].lattice
    uc_matrix = np.array([
        sc_lattice.matrix[0] / scaling[0],
        sc_lattice.matrix[1] / scaling[1],
        sc_lattice.matrix[2] / scaling[2]
    ])
    uc_lattice = Lattice(uc_matrix)
    
    n_atoms = len(unit_frac_coords)
    site_labels = -1 * np.ones(n_atoms, dtype=int)
    current_label = 0
    
    print("Grouping equivalent atoms...")
    grouped_indices = [] 
    for i in range(n_atoms):
        if site_labels[i] != -1: continue
        
        group = [i]
        site_labels[i] = current_label
        ref_coord = unit_frac_coords[i]
        
        for j in range(i + 1, n_atoms):
            if site_labels[j] != -1: continue
            
            diff = unit_frac_coords[j] - ref_coord
            diff -= np.round(diff) # Wrap to [-0.5, 0.5]
            cart_diff = np.dot(diff, uc_matrix)
            dist = np.linalg.norm(cart_diff)
            
            if dist < 0.5: # 0.5 Angstrom tolerance for "same site"
                site_labels[j] = current_label
                group.append(j)
        
        grouped_indices.append(group)
        current_label += 1
        
    print(f"Found {len(grouped_indices)} unique sites in the unit cell.")

    # 4. Calculate Pooled U_ij for each group
    # We need Cartesian Displacements from the trajectory
    sc_cart_coords = np.dot(unwrapped_frac, sc_lattice.matrix) # (T, N_sc, 3)
    
    final_sites = [] # (Species, FracCoords, U_ij)
    
    for group in grouped_indices:
        species = structures[0][group[0]].specie
        group_folded_fracs = unit_frac_coords[group]
        
        # Handle PBC averaging (if some are 0.99 and some 0.01)
        # Shift all relative to the first one
        ref = group_folded_fracs[0]
        diffs = group_folded_fracs - ref
        diffs -= np.round(diffs)
        avg_diff = np.mean(diffs, axis=0)
        site_avg_frac = (ref + avg_diff) % 1.0
        
        pooled_displacements = []
        for atom_idx in group:
            traj = sc_cart_coords[:, atom_idx, :]
            mean_pos = np.mean(traj, axis=0)
            displacement = traj - mean_pos
            pooled_displacements.append(displacement)
            
        pooled_displacements = np.vstack(pooled_displacements)
        cov = np.cov(pooled_displacements, rowvar=False)
        u_vals = [cov[0,0], cov[1,1], cov[2,2], cov[0,1], cov[0,2], cov[1,2]]
        final_sites.append({
            "species": species,
            "frac": site_avg_frac,
            "u": u_vals
        })
        
    return uc_lattice, final_sites

def write_averaged_cif(lattice, sites, filename="unit_cell_averaged.cif"):
    lines = []
    lines.append("data_unit_cell_averaged")
    lines.append("_audit_creation_method 'VASP Supercell -> Unit Cell Averaging'")
    
    abc = lattice.abc
    angles = lattice.angles
    lines.append(f"_cell_length_a    {abc[0]:.6f}")
    lines.append(f"_cell_length_b    {abc[1]:.6f}")
    lines.append(f"_cell_length_c    {abc[2]:.6f}")
    lines.append(f"_cell_angle_alpha {angles[0]:.6f}")
    lines.append(f"_cell_angle_beta  {angles[1]:.6f}")
    lines.append(f"_cell_angle_gamma {angles[2]:.6f}")
    lines.append("_symmetry_space_group_name_H-M 'P 1'")
    lines.append("loop_")
    lines.append("_symmetry_equiv_pos_as_xyz")
    lines.append("'x, y, z'")
    
    lines.append("loop_")
    lines.append("_atom_site_label")
    lines.append("_atom_site_type_symbol")
    lines.append("_atom_site_fract_x")
    lines.append("_atom_site_fract_y")
    lines.append("_atom_site_fract_z")
    
    labels = []
    for i, site in enumerate(sites):
        lbl = f"{site['species'].symbol}{i+1}"
        labels.append(lbl)
        f = site['frac']
        lines.append(f"{lbl} {site['species'].symbol} {f[0]:.6f} {f[1]:.6f} {f[2]:.6f}")
        
    lines.append("loop_")
    lines.append("_atom_site_aniso_label")
    lines.append("_atom_site_aniso_U_11")
    lines.append("_atom_site_aniso_U_22")
    lines.append("_atom_site_aniso_U_33")
    lines.append("_atom_site_aniso_U_12")
    lines.append("_atom_site_aniso_U_13")
    lines.append("_atom_site_aniso_U_23")
    
    for i, site in enumerate(sites):
        u = site['u']
        lines.append(f"{labels[i]} {u[0]:.6f} {u[1]:.6f} {u[2]:.6f} {u[3]:.6f} {u[4]:.6f} {u[5]:.6f}")
        
    with open(filename, "w") as f:
        f.write("\n".join(lines))
    print(f"Done! Saved to {filename}")



In [3]:
from pymatgen.io.vasp import Xdatcar
from pathlib import Path

BASE_PATH = Path("../")
MD_PATH = "md-mlff/300k"
SUPERCELL = [2, 2, 4]

systems = {
    "01_hzzrs3": "HzZrS3",
    "02_hzzrse3": "HzZrSe3",
    "03_hzhfs3": "HzHfS3",
    "04_hzhfse3": "HzHfSe3",
}

for system_key, system_name in systems.items():
    print(f"Processing {system_name}...")
    xdat = Xdatcar(BASE_PATH / system_key / MD_PATH /"XDATCAR")
    structures = xdat.structures
    reordered = reorder_trajectory(structures)
    unwrapped = unwrap_trajectory(reordered)
    uc_lattice, final_sites = fold_and_average_supercell(structures, unwrapped, SUPERCELL)
    write_averaged_cif(uc_lattice, final_sites, filename=BASE_PATH / "thermal-cifs" / f"{system_name}_averaged.cif")


Processing HzZrS3...
Reordering trajectory...
Unwrapping PBCs...
Folding 2x2x4 supercell...
Grouping equivalent atoms...
Found 48 unique sites in the unit cell.
Done! Saved to ../thermal-cifs/HzZrS3_averaged.cif
Processing HzZrSe3...
Reordering trajectory...
Unwrapping PBCs...
Folding 2x2x4 supercell...
Grouping equivalent atoms...
Found 48 unique sites in the unit cell.
Done! Saved to ../thermal-cifs/HzZrSe3_averaged.cif
Processing HzHfS3...
Reordering trajectory...
Unwrapping PBCs...
Folding 2x2x4 supercell...
Grouping equivalent atoms...
Found 48 unique sites in the unit cell.
Done! Saved to ../thermal-cifs/HzHfS3_averaged.cif
Processing HzHfSe3...
Reordering trajectory...
Unwrapping PBCs...
Folding 2x2x4 supercell...
Grouping equivalent atoms...
Found 48 unique sites in the unit cell.
Done! Saved to ../thermal-cifs/HzHfSe3_averaged.cif
